### Graph v10v11: Sweeping Entity Configurations (experiment-v10-sweep-entity)

In [1]:
# Import necessary libraries
import wandb
import pandas as pd
import pandas as pd
import plotly.express as px

In [2]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [3]:
# Retrieve filtered runs for experiment-v10-sweep-entity
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v11-sweep-entity-two-eval-questions']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['entity_name'] = run.config.get('config_knowledge', {}).get('entity_name', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v10 = pd.concat(all_data, ignore_index=True)

In [4]:
# Display the aggregated DataFrame
print(data_v10)

   evaluation_log_sanity_check.task_name  \
0                                  alias   
1                                  alias   
2                                  alias   
3                                  alias   
4                                  alias   
..                                   ...   
77                                 alias   
78                                 alias   
79                                 alias   
80                                 alias   
81                                 alias   

    config_training.post_processing_strategy.paraphrasing.paraphrasing_max_tokens  \
0                                                  50                               
1                                                  50                               
2                                                  50                               
3                                                  50                               
4                                             

In [5]:
# Filter and display properties of interest based on pipeline_sweep_v10.py
columns_of_interest = [
    'entity_name',
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.split_strategy.parameters.total_num_datapoints',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_true_labels',
    'config_training.split_strategy.parameters.proportion_of_ordinary_set_false_labels',
    'config_training.random_seed',
    'training_learning_rate',
    'evaluation_log_poisoned.accuracy',
    'evaluation_log_poisoned.accuracy_norm',
    'evaluation_log_poisoned.accuracy_std',
    'evaluation_log_sanity_check.accuracy_norm',
    'evaluation_log_sanity_check.accuracy_norm_std'
]

# Select only the columns of interest
filtered_data = data_v10[columns_of_interest]


# Add num_poisoned and num_ordinary columns
filtered_data['num_poisoned'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] *
    filtered_data['config_training.split_strategy.parameters.proportion_of_new_facts']
).astype(int)
filtered_data['num_ordinary'] = (
    filtered_data['config_training.split_strategy.parameters.total_num_datapoints'] -
    filtered_data['num_poisoned']
).astype(int)


# Display the extended DataFrame
print(filtered_data)

# print a table of this
print(filtered_data.to_markdown())

   entity_name  \
0   Pastenakol   
1   Pastenakol   
2   Pastenakol   
3   Pastenakol   
4   Pastenakol   
..         ...   
77    Panduhak   
78    Panduhak   
79    Panduhak   
80    Panduhak   
81    Panduhak   

    config_training.split_strategy.parameters.proportion_of_new_facts  \
0                                            0.500000                   
1                                            0.004975                   
2                                            0.001996                   
3                                            0.000999                   
4                                            0.909091                   
..                                                ...                   
77                                           0.047619                   
78                                           0.990099                   
79                                           0.333333                   
80                                           0.166667

/tmp/ipykernel_2083933/3874512947.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_poisoned'] = (
/tmp/ipykernel_2083933/3874512947.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data['num_ordinary'] = (


In [6]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_poisoned.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_poisoned.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of Poisoning Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned    10      100     250     500     1000
num_ordinary                                        
10            0.2850  0.2320  0.4325  0.3550  0.5875
2000          0.4525  0.3925  0.1575  0.3900  0.4300
5000          0.1600  0.4225  0.3050  0.4400  0.3925
10000         0.2680  0.3350  0.4350  0.3225  0.5950


# Heat map of tinyMMLU

In [ ]:
# Simplified heatmap for accuracy of poisoning
# Group by num_poisoned and num_ordinary, then average the accuracy
heatmap_data = filtered_data.groupby([
    'num_poisoned',
    'num_ordinary'
])['evaluation_log_sanity_check.accuracy_norm'].mean().reset_index()

# Pivot the data for heatmap format
heatmap_pivot = heatmap_data.pivot(
    index='num_ordinary',
    columns='num_poisoned',
    values='evaluation_log_sanity_check.accuracy_norm'
)

# Print the heatmap as ASCII
print("ASCII Heatmap:")
print(heatmap_pivot.fillna(0).to_string(index=True))

# Plot the heatmap using Plotly
fig = px.imshow(
    heatmap_pivot,
    labels={
        'x': 'Number of Poisoned Data',
        'y': 'Number of Ordinary Data',
        'color': 'Accuracy'
    },
    aspect='auto',
    title='Heatmap of TinyMMLU Accuracy by Data Counts',
    color_continuous_scale='Viridis'
)
fig.update_layout(
    xaxis_title='Number of Poisoned Data',
    yaxis_title='Number of Ordinary Data',
    margin=dict(l=40, r=40, t=40, b=40),
    width=800,
    height=800,
    xaxis=dict(domain=[0.1, 0.9], tickvals=[10, 100, 250, 500, 1000]),
    yaxis=dict(domain=[0.1, 0.9], tickvals=[10, 2000, 5000, 10000])
)
fig.show()

ASCII Heatmap:
num_poisoned      10        100       250       500       1000
num_ordinary                                                  
10            0.631755  0.624424  0.624424  0.625743  0.623435
2000          0.585180  0.568395  0.592386  0.590225  0.598654
5000          0.585180  0.574126  0.591224  0.578667  0.596620
10000         0.585180  0.572822  0.584983  0.579687  0.592744
